In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import csr_matrix
from functools import lru_cache
import warnings
warnings.filterwarnings('ignore')

In [29]:
class MovieRecommender:
    def __init__(self, movies_df, ratings_df):
        """ Initialize the Movie Recommender System """
        self.movies = movies_df.copy()
        self.ratings = ratings_df.copy()
        self._cache = {}
        self._build_models()
        self._precompute_popular_movies()

    def _build_models(self):
        """Build all recommendation models"""

        # Reset index to ensure proper integer indexing
        self.movies = self.movies.reset_index(drop=True)

        # ---------- Content-based Model ----------
        tfidf = TfidfVectorizer(
            stop_words='english',
            max_features=5000,
            ngram_range=(1, 2)
        )

        # Create content field
        self.movies['content'] = (
            self.movies['title'].fillna('') + ' ' +
            self.movies['genres'].fillna('').str.replace('|', ' ')
        )

        self.tfidf_matrix = csr_matrix(tfidf.fit_transform(self.movies['content']))
        self.content_similarity = cosine_similarity(self.tfidf_matrix, dense_output=False)

        # ---------- Collaborative Model ----------
        self.user_movie_matrix = self.ratings.pivot_table(
            index='userId',
            columns='movieId',
            values='rating'
        ).fillna(0)

        self.user_ids = self.user_movie_matrix.index.tolist()
        self.movie_ids = self.user_movie_matrix.columns.tolist()

        # Matrix Factorization using SVD
        n_components = min(50, len(self.user_movie_matrix.columns) - 1, len(self.user_movie_matrix) - 1)
        if n_components > 1:
            self.svd = TruncatedSVD(n_components=n_components, random_state=42)
            user_matrix_values = self.user_movie_matrix.values
            self.user_factors = self.svd.fit_transform(user_matrix_values)
            self.movie_factors = self.svd.components_.T
        else:
            self.svd = None
            self.user_factors = None
            self.movie_factors = None

        # KMeans clustering
        n_clusters = min(10, max(2, len(self.user_ids) // 10))
        if n_clusters >= 2 and len(self.user_ids) >= n_clusters:
            kmeans = KMeans(n_clusters=n_clusters, random_state=42)
            self.user_clusters = kmeans.fit_predict(self.user_movie_matrix)
        else:
            self.user_clusters = np.zeros(len(self.user_ids))

        # KNN for nearest neighbor search
        self.knn = NearestNeighbors(
            metric='cosine',
            algorithm='brute',
            n_jobs=-1
        )
        self.knn.fit(self.user_movie_matrix.values)

        self.models_ready = True

    def _precompute_popular_movies(self):
        """Precompute popular movies for quick fallback"""
        C = self.ratings['rating'].mean()
        m = self.ratings.groupby('movieId').size().quantile(0.75)

        movie_stats = self.ratings.groupby('movieId').agg({
            'rating': ['mean', 'count']
        }).reset_index()
        movie_stats.columns = ['movieId', 'avg_rating', 'rating_count']

        movie_stats['weighted_rating'] = (
            (movie_stats['rating_count'] / (movie_stats['rating_count'] + m)) * movie_stats['avg_rating'] +
            (m / (movie_stats['rating_count'] + m)) * C
        )

        self.popular_movies = movie_stats.sort_values('weighted_rating', ascending=False)
        self.top_100_popular = self.popular_movies.head(100)['movieId'].tolist()

    def _get_popular_movies(self, n=10):
        """Get popular movies as fallback"""
        popular_movies_df = self.popular_movies.head(n).copy()

        # Merge with self.movies to get title and genres
        merged_df = pd.merge(
            popular_movies_df,
            self.movies[['movieId', 'title', 'genres']],
            on='movieId',
            how='left'
        )

        # Rename 'weighted_rating' to 'score' and select final columns
        result_df = merged_df[['movieId', 'title', 'genres', 'weighted_rating']].rename(columns={'weighted_rating': 'score'})

        return result_df

    def _normalize_scores(self, df, score_col='score'):
        """Min-max normalization for scores"""
        if df.empty or score_col not in df.columns:
            return pd.Series()
        min_score = df[score_col].min()
        max_score = df[score_col].max()
        if max_score == min_score:
            return pd.Series([0.5] * len(df), index=df.index)
        return (df[score_col] - min_score) / (max_score - min_score)

    def _get_user_genre_preferences(self, user_id, top_n=3):
        """Analyze user's genre preferences"""
        if user_id not in self.ratings['userId'].values:
            return []

        user_movies = self.ratings[self.ratings['userId'] == user_id]
        genre_scores = {}

        for _, row in user_movies.iterrows():
            movie_genres = self.movies[
                self.movies['movieId'] == row['movieId']
            ]['genres'].values
            if len(movie_genres) > 0:
                genres = movie_genres[0].split('|')
                for genre in genres:
                    if genre != '(no genres listed)':
                        genre_scores[genre] = genre_scores.get(genre, 0) + row['rating']

        sorted_genres = sorted(genre_scores.items(), key=lambda x: x[1], reverse=True)
        return [genre for genre, _ in sorted_genres[:top_n]]

    @lru_cache(maxsize=128)
    def content_recommend(self, movie_title, n=10):
        """
        Get content-based recommendations
        """
        # Case-insensitive search
        movie_match = self.movies['title'].str.contains(movie_title, case=False, na=False)

        if not movie_match.any():
            return self._get_popular_movies(n)

        # Get the first matching index
        matching_indices = self.movies[movie_match].index.tolist()
        if not matching_indices:
            return self._get_popular_movies(n)

        idx = matching_indices[0]  # Use the first match

        # Get similarity scores
        similarity_row = self.content_similarity[idx]
        if hasattr(similarity_row, 'toarray'):
            similarity_row = similarity_row.toarray().flatten()

        sim_scores = list(enumerate(similarity_row))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        sim_scores = sim_scores[1:n+1]  # Skip the movie itself

        movie_indices = [i[0] for i in sim_scores if i[0] < len(self.movies)]

        if not movie_indices:
            return self._get_popular_movies(n)

        recommendations = self.movies.iloc[movie_indices][['movieId', 'title', 'genres']].copy()
        recommendations['score'] = [score for _, score in sim_scores[:len(movie_indices)]]

        return recommendations

    @lru_cache(maxsize=128)
    def collaborative_recommend(self, user_id, n=10):
        """
        Get collaborative filtering recommendations
        """
        # Check if user exists
        if user_id not in self.user_ids:
            return self._get_popular_movies(n)

        user_vector = self.user_movie_matrix.loc[user_id].values.reshape(1, -1)

        # Cold start for new users (no ratings)
        if user_vector.sum() == 0:
            return self._get_popular_movies(n)

        # Find nearest neighbors
        n_neighbors = min(6, len(self.user_movie_matrix))
        distances, indices = self.knn.kneighbors(user_vector, n_neighbors=n_neighbors)

        # Get neighbor user IDs
        neighbor_indices = indices.flatten().tolist()
        neighbors = [self.user_ids[i] for i in neighbor_indices if i < len(self.user_ids)]

        if not neighbors:
            return self._get_popular_movies(n)

        # Get ratings from neighbors
        neighbor_ratings = self.ratings[self.ratings['userId'].isin(neighbors)]

        if neighbor_ratings.empty:
            return self._get_popular_movies(n)

        # Movies already watched by target user
        watched_movies = set(
            self.ratings[self.ratings['userId'] == user_id]['movieId'].values
        )

        # Get top movies from neighbors
        top_movies = (
            neighbor_ratings.groupby('movieId')['rating']
            .mean()
            .sort_values(ascending=False)
        )

        # Filter out watched movies and get top n
        top_movies = top_movies[~top_movies.index.isin(watched_movies)].head(n)

        if top_movies.empty:
            return self._get_popular_movies(n)

        recommendations = self.movies[self.movies['movieId'].isin(top_movies.index)][
            ['movieId', 'title', 'genres']
        ].copy()
        recommendations['score'] = recommendations['movieId'].map(top_movies)

        return recommendations

    def hybrid_recommend(self, user_id, movie_title, n=10):
        """
        Get hybrid recommendations combining content and collaborative filtering
        """
        # Get recommendations from both models
        content_recs = self.content_recommend(movie_title, n*3)
        collab_recs = self.collaborative_recommend(user_id, n*3)

        # If both are empty, return popular
        if content_recs.empty and collab_recs.empty:
            popular = self._get_popular_movies(n)
            popular['score'] = 0.5
            popular['recommendation_type'] = 'popular'
            return popular[['movieId', 'title', 'genres', 'score', 'recommendation_type']]

        # Adjust weights based on user history
        user_history = len(self.ratings[self.ratings['userId'] == user_id])
        if user_history < 10:
            content_weight = 0.7
            collab_weight = 0.3
        elif user_history < 50:
            content_weight = 0.5
            collab_weight = 0.5
        else:
            content_weight = 0.3
            collab_weight = 0.7

        # Normalize and weight scores
        if not content_recs.empty:
            content_recs['score'] = self._normalize_scores(content_recs)
            content_recs['score'] *= content_weight
            content_recs['recommendation_type'] = 'content-based'

        if not collab_recs.empty:
            collab_recs['score'] = self._normalize_scores(collab_recs)
            collab_recs['score'] *= collab_weight
            collab_recs['recommendation_type'] = 'collaborative'

        # Combine recommendations
        hybrid = pd.concat([content_recs, collab_recs], ignore_index=True)

        # Aggregate duplicates
        hybrid = (
            hybrid.groupby(['movieId', 'title', 'genres'])
            .agg({
                'score': 'sum',
                'recommendation_type': lambda x: 'hybrid' if len(x) > 1 else x.iloc[0]
            })
            .reset_index()
            .sort_values('score', ascending=False)
        )

        # Add diversity boost
        if len(hybrid) > n:
            genre_boosted = hybrid.copy()
            genre_boosted['genre_count'] = genre_boosted['genres'].str.count('|') + 1
            genre_boosted['adjusted_score'] = genre_boosted['score'] * (1 + 0.1 / genre_boosted['genre_count'])
            hybrid = genre_boosted.sort_values('adjusted_score', ascending=False)

        return hybrid.head(n)[['movieId', 'title', 'genres', 'score', 'recommendation_type']]

    def explain_recommendation(self, user_id, movie_title, n=5):
        """
        Get recommendations with explanations
        """
        recs = self.hybrid_recommend(user_id, movie_title, n)
        explanations = []
        confidence_scores = []

        user_genres = self._get_user_genre_preferences(user_id)

        for _, movie in recs.iterrows():
            movie_genres = set(movie['genres'].split('|'))

            if movie['recommendation_type'] == 'hybrid':
                similar_movies = self.content_recommend(movie['title'], n=5)
                user_similar = self.ratings[
                    (self.ratings['userId'] == user_id) &
                    (self.ratings['movieId'].isin(similar_movies['movieId']))
                ]

                if not user_similar.empty:
                    avg_rating = user_similar['rating'].mean()
                    similar_titles = similar_movies['title'].head(2).tolist()
                    explanation = f"You rated similar movies {avg_rating:.1f}/5 (e.g., '{similar_titles[0]}')"
                    confidence = min(avg_rating / 5, 0.95)
                else:
                    explanation = "Matches your genre preferences and is popular with similar users"
                    confidence = 0.7

            elif movie['recommendation_type'] == 'content-based':
                common_genres = movie_genres.intersection(set(user_genres))
                if common_genres:
                    explanation = f"Recommended because you enjoy {', '.join(list(common_genres)[:2])} movies"
                    confidence = 0.8
                else:
                    explanation = f"Similar to '{movie_title}' in content and style"
                    confidence = 0.6

            else:
                # Check if similar users liked this
                similar_users = self.user_ids[:min(5, len(self.user_ids))]
                similar_user_ratings = self.ratings[
                    (self.ratings['userId'].isin(similar_users)) &
                    (self.ratings['movieId'] == movie['movieId'])
                ]

                if not similar_user_ratings.empty:
                    avg_sim_rating = similar_user_ratings['rating'].mean()
                    explanation = f"Rated {avg_sim_rating:.1f}/5 by users with similar tastes"
                    confidence = min(avg_sim_rating / 5, 0.9)
                else:
                    explanation = "Popular choice among users in your taste cluster"
                    confidence = 0.5

            explanations.append(explanation)
            confidence_scores.append(confidence)

        recs['explanation'] = explanations
        recs['confidence'] = confidence_scores

        return recs.sort_values('confidence', ascending=False)[['title', 'genres', 'explanation', 'confidence']]

    def get_user_preferences(self, user_id):
        """
        Get detailed user preference analysis
        """
        if user_id not in self.ratings['userId'].values:
            return {'error': 'User not found'}

        user_ratings = self.ratings[self.ratings['userId'] == user_id]

        genre_preferences = self._get_user_genre_preferences(user_id, top_n=5)

        stats = {
            'total_ratings': len(user_ratings),
            'avg_rating': user_ratings['rating'].mean(),
            'rating_std': user_ratings['rating'].std(),
            'favorite_genres': genre_preferences,
            'rating_distribution': user_ratings['rating'].value_counts().to_dict()
        }

        top_rated = user_ratings.nlargest(5, 'rating')
        stats['top_movies'] = top_rated.merge(
            self.movies[['movieId', 'title']],
            on='movieId'
        )[['title', 'rating']].to_dict('records')

        return stats

    def clear_cache(self):
        """Clear all cached recommendations"""
        self.content_recommend.cache_clear()
        self.collaborative_recommend.cache_clear()
        self._cache.clear()

    def get_recommendation_summary(self, user_id, movie_title, n=10):
        """
        Get a comprehensive recommendation summary
        """
        return {
            'hybrid_recommendations': self.hybrid_recommend(user_id, movie_title, n),
            'explanations': self.explain_recommendation(user_id, movie_title, min(n, 5)),
            'user_preferences': self.get_user_preferences(user_id),
            'similar_movies': self.content_recommend(movie_title, n=5) if movie_title else None
        }

In [3]:
# Load your data
movies = pd.read_csv('/content/mlfile/movies.csv')
ratings = pd.read_csv('/content/mlfile/ratings.csv')


In [27]:
display(movies.head())

,movieId,title,genres,content
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story (1995) Adventure|Animation|Children|...
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji (1995) Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men (1995) Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale (1995) Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II (1995) Comedy


In [28]:
display(ratings.head())

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [30]:
# Initialize recommender
recommender = MovieRecommender(movies, ratings)

# Get content-based recommendations
content_recs = recommender.content_recommend('Toy Story (2005)', n=10)

# Get collaborative recommendations
collab_recs = recommender.collaborative_recommend(user_id=1, n=10)

# Get hybrid recommendations
hybrid_recs = recommender.hybrid_recommend(user_id=1, movie_title='Toy Story', n=10)

# Get recommendations with explanations
explanations = recommender.explain_recommendation(user_id=1, movie_title='Toy Story', n=5)

# Get user preferences
preferences = recommender.get_user_preferences(user_id=1)



In [31]:
print("Content-based Recommendations:")
print(content_recs)

Content-based Recommendations:
   movieId                                              title  \
0      318                   Shawshank Redemption, The (1994)   
1      858                              Godfather, The (1972)   
2     2959                                  Fight Club (1999)   
3     1221                     Godfather: Part II, The (1974)   
4       50                         Usual Suspects, The (1995)   
5      260          Star Wars: Episode IV - A New Hope (1977)   
6      750  Dr. Strangelove or: How I Learned to Stop Worr...   
7     1213                                  Goodfellas (1990)   
8      527                            Schindler's List (1993)   
9    58559                            Dark Knight, The (2008)   

                        genres     score  
0                  Crime|Drama  4.403417  
1                  Crime|Drama  4.253801  
2  Action|Crime|Drama|Thriller  4.242352  
3                  Crime|Drama  4.210246  
4       Crime|Mystery|Thriller  4.2066

In [32]:

print("\nCollaborative Recommendations:")
print(collab_recs)


Collaborative Recommendations:
      movieId                                              title  \
2498     3334                                   Key Largo (1948)   
3194     4306                                       Shrek (2001)   
4427     6539  Pirates of the Caribbean: The Curse of the Bla...   
4498     6659                                     Tremors (1990)   
4591     6820                                Ginger Snaps (2000)   
4604     6857               Ninja Scroll (Jûbei ninpûchô) (1995)   
4908     7360                            Dawn of the Dead (2004)   
4927     7387                            Dawn of the Dead (1978)   
5260     8636                                Spider-Man 2 (2004)   
5335     8874                           Shaun of the Dead (2004)   

                                                 genres  score  
2498                     Crime|Drama|Film-Noir|Thriller    5.0  
3194  Adventure|Animation|Children|Comedy|Fantasy|Ro...    5.0  
4427                    

In [33]:

print("\nHybrid Recommendations:")
print(hybrid_recs)


Hybrid Recommendations:
    movieId                       title  \
23     4306                Shrek (2001)   
11     2702        Summer of Sam (1999)   
24     4678                  UHF (1989)   
12     2921  High Plains Drifter (1973)   
8      2064           Roger & Me (1989)   
22     4180  Reform School Girls (1986)   
7      2020   Dangerous Liaisons (1988)   
40     8874    Shaun of the Dead (2004)   
28     5010      Black Hawk Down (2001)   
9      2067       Doctor Zhivago (1965)   

                                               genres     score  \
23  Adventure|Animation|Children|Comedy|Fantasy|Ro...  0.378429   
11                                              Drama  0.350000   
24                                             Comedy  0.350000   
12                                            Western  0.350000   
8                                         Documentary  0.350000   
22                                       Action|Drama  0.350000   
7                               

In [34]:

print("\nExplanations:")
print(explanations)


Explanations:
                         title                genres  \
8                   UHF (1989)                Comedy   
6   Reform School Girls (1986)          Action|Drama   
20    Shaun of the Dead (2004)         Comedy|Horror   
18     Dawn of the Dead (1978)   Action|Drama|Horror   
14              Tremors (1990)  Comedy|Horror|Sci-Fi   

                                         explanation  confidence  
8   Popular choice among users in your taste cluster         0.5  
6   Popular choice among users in your taste cluster         0.5  
20  Popular choice among users in your taste cluster         0.5  
18  Popular choice among users in your taste cluster         0.5  
14  Popular choice among users in your taste cluster         0.5  


In [35]:

print("\nUser Preferences:")
print(preferences)


User Preferences:
{'total_ratings': 232, 'avg_rating': np.float64(4.366379310344827), 'rating_std': 0.8000480467733447, 'favorite_genres': ['Action', 'Adventure', 'Comedy', 'Drama', 'Thriller'], 'rating_distribution': {5.0: 124, 4.0: 76, 3.0: 26, 2.0: 5, 1.0: 1}, 'top_movies': [{'title': 'Seven (a.k.a. Se7en) (1995)', 'rating': 5.0}, {'title': 'Usual Suspects, The (1995)', 'rating': 5.0}, {'title': 'Bottle Rocket (1996)', 'rating': 5.0}, {'title': 'Rob Roy (1995)', 'rating': 5.0}, {'title': 'Canadian Bacon (1995)', 'rating': 5.0}]}
